In [4]:
import os
import torch
import subprocess

# Check GPU
print("="*50)
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"VRAM per GPU: {total_mem / 1e9:.1f} GB")

# Check disk space
result = subprocess.run(['df', '-h', '/kaggle/working'], 
                       capture_output=True, text=True)
print("\nDisk Space:")
print(result.stdout)

# Check RAM
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("RAM:")
print(result.stdout)

GPU Available: True
GPU Name: NVIDIA H100 80GB HBM3
GPU Count: 1
VRAM per GPU: 85.0 GB

Disk Space:

RAM:
               total        used        free      shared  buff/cache   available
Mem:           230Gi       7.6Gi       182Gi       2.3Mi        41Gi       222Gi
Swap:           31Gi          0B        31Gi



In [5]:
# Install all required packages
# Kaggle already has torch, torchvision, numpy pre-installed

!pip install -q \
    omegaconf \
    h5py \
    hdf5plugin \
    zstandard \
    huggingface_hub \
    tqdm \
    scikit-learn \
    imageio \
    wandb

print("All packages installed!")

# Verify key imports
import torch
import h5py
import omegaconf
import huggingface_hub
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

All packages installed!
PyTorch: 2.8.0+cu128
CUDA: 12.8


In [6]:
import os
import sys

# If using GitHub method:
!git clone https://github.com/krishbansal-2205/lewm-pushT.git /teamspace/studios/this_studio/lewm-pushT
# %cd /kaggle/working/lewm
%cd /teamspace/studios/this_studio/lewm-pushT
sys.path.insert(0,'/teamspace/studios/this_studio/lewm-pushT')
# sys.path.insert(0, '/kaggle/working/lewm')

# Verify structure
import os
for root, dirs, files in os.walk('/teamspace/studios/this_studio/lewm-pushT'):
    # Skip hidden folders
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace('/teamspace/studios/this_studio/lewm-pushT', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

fatal: destination path '/teamspace/studios/this_studio/lewm-pushT' already exists and is not an empty directory.


⚡️ Tip	Connect GitHub to Studios: https://lightning.ai/krishbansal10st/home?settings=integrations

/teamspace/studios/this_studio/lewm-pushT
lewm-pushT/
  .gitignore
  README.md
  conftest.py
  evaluate.py
  requirements.txt
  train.py
  configs/
    kaggle_t4.yaml
    pusht.yaml
  data/
    __init__.py
    download.py
    __pycache__/
  dataset/
    pusht_expert_train.h5
  evaluation/
    __init__.py
    eval.py
  models/
    __init__.py
    encoder.py
    lewm.py
    predictor.py
    __pycache__/
  planning/
    __init__.py
    cem.py
  tests/
    __init__.py
    test_cem.py
    test_dataset.py
    test_encoder.py
    test_predictor.py
    test_sigreg.py
  training/
    __init__.py
    dataset.py
    sigreg.py
    train.py
    __pycache__/
  visualization/
    __init__.py
    visualize.py


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
import os

# Set data directory
os.environ["LEWM_DATA_DIR"] = "/teamspace/studios/this_studio/lewm-pushT/dataset"
os.makedirs("/teamspace/studios/this_studio/lewm-pushT/dataset", exist_ok=True)

# Check if dataset already cached (useful if running multiple times)
h5_path = "/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5"

if os.path.exists(h5_path):
    size_gb = os.path.getsize(h5_path) / 1e9
    print(f"Dataset already exists: {size_gb:.2f} GB")
else:
    print("Downloading dataset from HuggingFace...")
    print("This will take 10-20 minutes (12GB compressed)")
    
    # Set HuggingFace token if needed (for private repos)
    # os.environ["HUGGINGFACE_TOKEN"] = "your_token_here"
    
    !python -m data.download --data-dir /teamspace/studios/this_studio/lewm-pushT/dataset
    
    if os.path.exists(h5_path):
        size_gb = os.path.getsize(h5_path) / 1e9
        print(f"Download complete: {size_gb:.2f} GB")
    else:
        print("ERROR: Download failed!")

Dataset already exists: 46.30 GB


In [8]:
import h5py
import numpy as np

h5_path = "/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5"

with h5py.File(h5_path, 'r') as f:
    print("Dataset structure:")
    print("="*40)
    for key in f.keys():
        dataset = f[key]
        print(f"  {key}:")
        print(f"    Shape: {dataset.shape}")
        print(f"    Dtype: {dataset.dtype}")
        if hasattr(dataset, 'nbytes'):
            print(f"    Size:  {dataset.nbytes / 1e9:.2f} GB")
    
    # Show sample
    print("\nSample observation shape:", f['observations'][0].shape)
    print("Sample action:", f['actions'][0])
    print("Total samples:", f['observations'].shape[0])

Dataset structure:
  action:
    Shape: (2336736, 2)
    Dtype: float32
    Size:  0.02 GB
  ep_len:
    Shape: (18685,)
    Dtype: int32
    Size:  0.00 GB
  ep_offset:
    Shape: (18685,)
    Dtype: int64
    Size:  0.00 GB
  episode_idx:
    Shape: (2336736,)
    Dtype: int64
    Size:  0.02 GB
  pixels:
    Shape: (2336736, 224, 224, 3)
    Dtype: uint8
    Size:  351.74 GB
  proprio:
    Shape: (2336736, 4)
    Dtype: float32
    Size:  0.04 GB
  state:
    Shape: (2336736, 7)
    Dtype: float32
    Size:  0.07 GB
  step_idx:
    Shape: (2336736,)
    Dtype: int64
    Size:  0.02 GB


KeyError: "Unable to synchronously open object (object 'observations' doesn't exist)"

In [4]:
# Write optimized config for Kaggle T4
config_content = """
# LeWM Kaggle T4 Configuration

# Data
data_dir: /teamspace/studios/this_studio/lewm-pushT/dataset
image_size: 224
train_split: 0.9
augmentation: true
num_workers: 8

# Model
latent_dim: 192
encoder_channels: [32, 64, 128, 256]
predictor_hidden: [512, 512, 512]
dropout: 0.1

# Training - optimized for T4 16GB VRAM
batch_size: 2048
epochs: 100
lr: 3e-4
weight_decay: 1e-4
grad_clip: 1.0
lambda_reg: 0.1
early_stopping_patience: 20
checkpoint_dir: /teamspace/studios/this_studio/lewm-pushT/checkpoints

# SIGReg
sigreg_num_projections: 64

# Planner
cem_n_samples: 512
cem_top_k: 64
cem_n_iters: 5
cem_horizon: 10
action_dim: 2
action_low: -1.0
action_high: 1.0

# Evaluation
n_eval_episodes: 100
max_steps_per_episode: 50
success_threshold: 0.15

# Misc
seed: 42
device: cuda
log_every: 50
"""

os.makedirs('/teamspace/studios/this_studio/lewm-pushT/configs', exist_ok=True)
with open('/teamspace/studios/this_studio/lewm-pushT/configs/kaggle_t4.yaml', 'w') as f:
    f.write(config_content)

print("Config saved!")
print(config_content)

Config saved!

# LeWM Kaggle T4 Configuration

# Data
data_dir: /teamspace/studios/this_studio/lewm-pushT/dataset
image_size: 224
train_split: 0.9
augmentation: true
num_workers: 8

# Model
latent_dim: 192
encoder_channels: [32, 64, 128, 256]
predictor_hidden: [512, 512, 512]
dropout: 0.1

# Training - optimized for T4 16GB VRAM
batch_size: 2048
epochs: 100
lr: 3e-4
weight_decay: 1e-4
grad_clip: 1.0
lambda_reg: 0.1
early_stopping_patience: 20
checkpoint_dir: /teamspace/studios/this_studio/lewm-pushT/checkpoints

# SIGReg
sigreg_num_projections: 64

# Planner
cem_n_samples: 512
cem_top_k: 64
cem_n_iters: 5
cem_horizon: 10
action_dim: 2
action_low: -1.0
action_high: 1.0

# Evaluation
n_eval_episodes: 100
max_steps_per_episode: 50
success_threshold: 0.15

# Misc
seed: 42
device: cuda
log_every: 50



In [5]:
# Quick sanity check - make sure everything works
import sys
sys.path.insert(0, '/teamspace/studios/this_studio/lewm-pushT')

import torch
from omegaconf import OmegaConf
from models.lewm import LeWM
from training.dataset import get_dataloaders

# Load config
config = OmegaConf.load('/teamspace/studios/this_studio/lewm-pushT/configs/kaggle_t4.yaml')
device = torch.device('cuda')

# Build model
print("Building model...")
model = LeWM(config).to(device)
params = model.count_parameters()
print(f"Parameters: {params['total']:,}")
print(f"  Encoder:   {params['encoder']:,}")
print(f"  Predictor: {params['predictor']:,}")

# Test forward pass with dummy data
print("\nTesting forward pass...")
obs = torch.randn(4, 3, 224, 224).to(device)
action = torch.randn(4, 2).to(device)
next_obs = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    total_loss, pred_loss, reg_loss = model.compute_loss(obs, action, next_obs)

print(f"Total loss:  {total_loss.item():.4f}")
print(f"Pred loss:   {pred_loss.item():.4f}")
print(f"Reg loss:    {reg_loss.item():.4f}")
print("\nForward pass OK!")

# Test memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM used: {mem_used:.2f} / {mem_total:.1f} GB")

Building model...
Parameters: 10,849,792
  Encoder:   10,023,744
  Predictor: 826,048

Testing forward pass...
Total loss:  2.3903
Pred loss:   2.1937
Reg loss:    1.9662

Forward pass OK!

VRAM used: 0.13 / 85.0 GB


In [6]:
from training.dataset import get_dataloaders
import time

print("Loading dataset...")
t = time.time()

train_loader, val_loader, dataset = get_dataloaders(
    h5_path="/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5",
    batch_size=512,
    train_split=0.9,
    augmentation=True,
    num_workers=4,
    image_size=224,
    seed=42,
)

print(f"Dataset loaded in {time.time()-t:.1f}s")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# Test one batch load speed
t = time.time()
batch = next(iter(train_loader))
print(f"\nFirst batch loaded in {time.time()-t:.2f}s")
print(f"obs shape:      {batch['obs'].shape}")
print(f"action shape:   {batch['action'].shape}")
print(f"next_obs shape: {batch['next_obs'].shape}")

Loading dataset...
Dataset loaded: 2318051 total samples
  Train: 2086245 samples (2086245 / 2318051)
  Val:   231806 samples (231806 / 2318051)
  Batch size: 512
  Augmentation: True
Dataset loaded in 1.1s
Train batches: 4074
Val batches:   453

First batch loaded in 10.27s
obs shape:      torch.Size([512, 3, 224, 224])
action shape:   torch.Size([512, 2])
next_obs shape: torch.Size([512, 3, 224, 224])


In [ ]:
import os
import sys
sys.path.insert(0, '/teamspace/studios/this_studio/lewm-pushT')
os.chdir('/teamspace/studios/this_studio/lewm-pushT')

from omegaconf import OmegaConf
from models.lewm import LeWM
from training.train import set_seed, train_lewm
from training.dataset import get_dataloaders
import torch

# Load config
config = OmegaConf.load('configs/kaggle_t4.yaml')
device = torch.device('cuda')

# Set seed
set_seed(config.seed)

# Create checkpoint dir
os.makedirs(config.checkpoint_dir, exist_ok=True)

# Load data
print("Loading data...")
train_loader, val_loader, dataset = get_dataloaders(
    h5_path=config.data_dir + "/pusht_expert_train.h5",
    batch_size=config.batch_size,
    train_split=config.train_split,
    augmentation=config.augmentation,
    num_workers=config.num_workers,
    image_size=config.image_size,
    seed=config.seed,
)

# Build model
print("Building model...")
model = LeWM(config).to(device)
print(f"Parameters: {model.count_parameters()['total']:,}")

# Train
print("\nStarting training...")
history = train_lewm(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    checkpoint_dir=config.checkpoint_dir,
)

print("\nTraining complete!")

Loading data...
Dataset loaded: 2318051 total samples
  Train: 2086245 samples (2086245 / 2318051)
  Val:   231806 samples (231806 / 2318051)
  Batch size: 2048
  Augmentation: True
Building model...
Parameters: 10,849,792

Starting training...

Starting LeWM Training
  Epochs: 100
  Batch size: 2048
  LR: 0.0003
  Lambda_reg: 0.1
  Device: cuda
  Checkpoints: /teamspace/studios/this_studio/lewm-pushT/checkpoints




⚠ WARNING: Possible representation collapse detected!
  Latent std = 0.007678 (threshold: 0.01)
  Consider increasing lambda_reg (currently 0.1)

  Auto-increased lambda_reg to 0.2
Epoch   1/100 | Train: 0.0204 (pred=0.0166, reg=0.0385) | Val: 0.0018 (pred=0.0001) | Latent std: 0.0077 | LR: 3.00e-04 | Time: 2283.5s
  ✓ New best model saved (val_pred=0.0001)


Epoch   2/100:   0%|          | 0/1018 [00:00<?, ?it/s]

In [ ]:
import torch
import matplotlib.pyplot as plt
import os

# Load history
history_path = "/kaggle/working/checkpoints/training_history.pt"
history = torch.load(history_path, weights_only=False)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs = range(1, len(history['train_loss']) + 1)

# Total loss
axes[0,0].plot(epochs, history['train_loss'], label='Train', color='blue')
axes[0,0].plot(epochs, history['val_loss'], label='Val', color='red')
axes[0,0].set_title('Total Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Prediction loss
axes[0,1].plot(epochs, history['train_pred_loss'], label='Train', color='green')
axes[0,1].plot(epochs, history['val_pred_loss'], label='Val', color='orange')
axes[0,1].set_title('Prediction Loss (MSE)')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# SIGReg loss
axes[1,0].plot(epochs, history['train_reg_loss'], label='Train', color='purple')
axes[1,0].set_title('SIGReg Loss')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Latent std
axes[1,1].plot(epochs, history['latent_std'], color='teal')
axes[1,1].axhline(y=0.01, color='red', linestyle='--', label='Collapse threshold')
axes[1,1].set_title('Latent Std (Collapse Monitor)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('LeWM Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved training_curves.png")